# Contract

A contract is the base concept from which you can calculate energy costs.
It consists of 4 components:
- a provider tariff
- a distributor tariff
- fees (which are essentially a government tariff)
- a tax rate (a percentage applied to all non government tariff cost)

Given a contract and consumption data, you can calculate the cost of that consumption under the terms of the contract.

In [ ]:
import pandas as pd

from energy_cost import Contract, Tariff
from energy_cost.data.be import distributors, fees, tax_rate

contract = Contract(
    provider=Tariff.from_yaml("../examples/tariffs/fixed.yml"),
    distributor=distributors["fluvius_imewo"],
    fees=fees["flanders_residential"],
    tax_rate=tax_rate,
)

consumption = pd.DataFrame({
    "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2024-03-01T00:00:00+01:00", freq="15min"),
    "value": 0.0002
})

contract.calculate_cost(consumption)

timestamp    provider               distributor             \
                            consumption         total consumption              
                                 energy  total  total      energy      total   
0 2024-01-01 00:00:00+01:00       59.52  59.52  59.52   23.676520  23.676520   
1 2024-02-01 00:00:00+01:00       55.68  55.68  55.68   22.149003  22.149003   
2 2024-03-01 00:00:00+01:00        0.02   0.02   0.02    0.007956   0.007956   

                                                      taxes      total  
   capacity            fixed                total     total      total  
      total data_manangement     total      total     total      total  
0  8.209764         1.209508  1.209508  33.095793  5.556948  98.172740  
1  8.209764         1.131475  1.131475  31.490243  5.230215  92.400457  
2  8.209764         0.000406  0.000406   8.218127  0.494288   8.732414

> Note: most tariffs are based on indexes, make sure to define all required indexes before calculating costs. See the [index documentation](./index.ipynb) for more details. If you are definingn your own tariffs, also have a look at the [tariff documentation](./tariff.ipynb) for details on how to implement tariffs.

By default the cost is calculated for each month, but you can specify a different resolution if you want to calculate costs for different time periods eg yearly.

In [5]:
import isodate

consumption = pd.DataFrame({
    "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
    "value": 0.0002
})

contract.calculate_cost(consumption, resolution=isodate.Duration(years=1))

timestamp    provider                                \
                            consumption              fixed              
                                 energy   total  fixed_fee      total   
0 2024-01-01 00:00:00+01:00      702.72  702.72    0.00000    0.00000   
1 2025-01-01 00:00:00+01:00      630.72  630.72  120.00000  120.00000   
2 2026-01-01 00:00:00+01:00      135.96  135.96   30.00504   30.00504   

             distributor                                                      \
       total consumption                capacity            fixed              
       total      energy       total       total data_manangement      total   
0  702.72000  279.535692  279.535692   98.517173        14.280000  14.280000   
1  750.72000  412.792925  412.792925  133.105596        17.510000  17.510000   
2  165.96504   59.240491   59.240491   33.875614         2.885852   2.885852   

                   taxes        total  
        total      total        total  
        total      total        total  
0  392.332864  65.703172  1160.756036  
1  563.408521  78.847711  1392.976232  
2   96.001957  15.718020   277.685017

By defaault start and end are set to the first and last timestamp in the data, but you can specify a different start and end if you want to calculate costs for a different time period than the one covered by the data.

This is usefull for the Flemish capacity tariff that uses a rolling average of the consumption of the last 12 months to determine the cost for the next month. In this case you would need to provide data for at least 12 months before the start of the period you want to calculate costs for.

In [18]:
import datetime as dt
import pytz

consumption = pd.DataFrame({
    "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
    "value": [0.0002] * 4 * 24 * 365 + [0.0003] * 4 * 24 * 365 + [0.0003] * 4 * 24 * 60 + [0.0005]
})

contract.calculate_cost(consumption, start=dt.datetime(2025, 1, 1, tzinfo=pytz.timezone("Europe/Brussels")), end=dt.datetime(2025, 12, 31, tzinfo=pytz.timezone("Europe/Brussels")))

timestamp    provider                              \
                             consumption            fixed              
                                  energy total  fixed_fee      total   
0  2024-12-31 23:00:00+00:00         0.0   0.0        NaN        NaN   
1  2024-12-31 23:42:00+00:00         NaN   NaN  10.001008  10.001008   
2  2025-01-31 23:00:00+00:00         0.0   0.0        NaN        NaN   
3  2025-01-31 23:42:00+00:00         NaN   NaN   9.998992   9.998992   
4  2025-02-28 23:00:00+00:00         0.0   0.0        NaN        NaN   
5  2025-02-28 23:42:00+00:00         NaN   NaN  10.000314  10.000314   
6  2025-03-31 22:42:00+00:00         NaN   NaN   9.999686   9.999686   
7  2025-03-31 23:00:00+00:00         0.0   0.0        NaN        NaN   
8  2025-04-30 22:42:00+00:00         NaN   NaN  10.000314  10.000314   
9  2025-04-30 23:00:00+00:00         0.0   0.0        NaN        NaN   
10 2025-05-31 22:42:00+00:00         NaN   NaN   9.999686   9.999686   
11 2025-05-31 23:00:00+00:00         0.0   0.0        NaN        NaN   
12 2025-06-30 22:42:00+00:00         NaN   NaN  10.000000  10.000000   
13 2025-06-30 23:00:00+00:00         0.0   0.0        NaN        NaN   
14 2025-07-31 22:42:00+00:00         NaN   NaN  10.000314  10.000314   
15 2025-07-31 23:00:00+00:00         0.0   0.0        NaN        NaN   
16 2025-08-31 22:42:00+00:00         NaN   NaN   9.999686   9.999686   
17 2025-08-31 23:00:00+00:00         0.0   0.0        NaN        NaN   
18 2025-09-30 22:42:00+00:00         NaN   NaN  10.000314  10.000314   
19 2025-09-30 23:00:00+00:00         0.0   0.0        NaN        NaN   
20 2025-10-31 23:00:00+00:00         0.0   0.0        NaN        NaN   
21 2025-10-31 23:42:00+00:00         NaN   NaN   9.999686   9.999686   
22 2025-11-30 23:00:00+00:00         0.0   0.0        NaN        NaN   
23 2025-11-30 23:42:00+00:00         NaN   NaN  19.668011  19.668011   

              distributor                                               \
        total consumption         capacity            fixed              
        total      energy total      total data_manangement      total   
0    0.000000         0.0   0.0        NaN              NaN        NaN   
1   10.001008         NaN   NaN        NaN         1.487151   1.487151   
2    0.000000         0.0   0.0  11.092133              NaN        NaN   
3    9.998992         NaN   NaN        NaN         1.343233   1.343233   
4    0.000000         0.0   0.0  11.092133              NaN        NaN   
5   10.000314         NaN   NaN        NaN         1.487151   1.487151   
6    9.999686         NaN   NaN        NaN         1.439178   1.439178   
7    0.000000         0.0   0.0  11.092133              NaN        NaN   
8   10.000314         NaN   NaN        NaN         1.487151   1.487151   
9    0.000000         0.0   0.0  11.092133              NaN        NaN   
10   9.999686         NaN   NaN        NaN         1.439178   1.439178   
11   0.000000         0.0   0.0  11.092133              NaN        NaN   
12  10.000000         NaN   NaN        NaN         1.487151   1.487151   
13   0.000000         0.0   0.0  11.092133              NaN        NaN   
14  10.000314         NaN   NaN        NaN         1.487151   1.487151   
15   0.000000         0.0   0.0  11.092133              NaN        NaN   
16   9.999686         NaN   NaN        NaN         1.439178   1.439178   
17   0.000000         0.0   0.0  11.092133              NaN        NaN   
18  10.000314         NaN   NaN        NaN        18.997151  18.997151   
19   0.000000         0.0   0.0  11.092133              NaN        NaN   
20   0.000000         0.0   0.0  11.092133              NaN        NaN   
21   9.999686         NaN   NaN        NaN         1.439178   1.439178   
22   0.000000         0.0   0.0  11.092133              NaN        NaN   
23  19.668011         NaN   NaN        NaN        18.947779  18.947779   

                  taxes      total  
        total     total      total  
 